# Volume 11, Chapter 11: Random matrices II, unitary matrices and the zeros of zeta (Python companion)

Exercises first, solutions at the end. Haar random unitary matrices, the coin flips hidden in an eigenvalue count, the logarithmic number variance, traces of powers, the moments of the characteristic polynomial, and then **ten thousand zeros of the Riemann zeta function**, computed from scratch with the Riemann–Siegel formula, compared with the sine kernel. The last problem tests the Keating–Snaith philosophy on the values of $\zeta$ itself. Requires `numpy` and `scipy`.

In [1]:
import math
import numpy as np
from scipy import special
rng = np.random.default_rng(1111)

def haar(n):
    # Volume 8, Theorem 8.7: QR of a complex Gaussian matrix, with the phases of R removed
    G = (rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))) / math.sqrt(2)
    Q, R = np.linalg.qr(G); d = np.diag(R)
    return Q * (d / abs(d))[None, :]

def angles(U):
    return np.sort(np.mod(np.angle(np.linalg.eigvals(U)), 2 * np.pi))

sinc = np.sinc   # numpy's sinc is sin(pi x)/(pi x), the function S of the chapter

## Exercises

**P11.1 (★) The coin flips.** For $n=40$ and the arc $D=[0,\frac\pi2)$, compute the eigenvalues $p_j$ of the matrix $M_D$ of Proposition 11.1 and the law of $B_1+\dots+B_n$ (a Poisson binomial law). Compare it with the empirical law of $N(D)$ over $20000$ Haar matrices, and with the binomial law of $40$ independent uniform points.

**P11.2 (★★) Rigidity.** For $n=400$, compute the exact number variance (11.1) for arcs containing on average $L=1,2,4,\dots,200$ eigenvalues and compare it with $\frac1{\pi^2}\log L$ (estimate the constant) and with the binomial variance $L(1-\frac Ln)$. Check one value by simulation.

**P11.3 (★★) Traces of powers.** For $n=12$, estimate $\mathbb E|\operatorname{tr}U^j|^2$ for $j=1,\dots,30$ and compare with $\min(j,n)$ (Proposition 11.12). Diaconis and Shahshahani proved that $\operatorname{tr}U$ is asymptotically a standard complex Gaussian: compare the law of $|\operatorname{tr}U|^2$ for $n=12$ with the exponential law of mean $1$.

**P11.4 (★★) Moments of the characteristic polynomial.** Estimate $\mathbb E|\det(I-U)|^{2m}$ for $m=1,2$ and $n=10$ from Haar matrices, and for $m=1,2,3$ and $n=10,100$ by sampling the independent factors $\gamma_k$ of Theorem 11.9 directly. Compare with Theorem 11.10. Why is $m=3$ so much harder to estimate than $m=1$?

**P11.5 (★★★) Odlyzko's experiment.** Compute all the zeros of $\zeta(\frac12+it)$ with $10^6<t<10^6+5000$, using the Riemann–Siegel function $Z(t)=e^{i\vartheta(t)}\zeta(\frac12+it)$, which is real: locate its sign changes and refine them by bisection. Check that none is missed by comparing their number with $\frac1\pi(\vartheta(T_1)-\vartheta(T_0))$ (Riemann–von Mangoldt). Rescale the zeros to mean spacing $1$ and compare (a) the spacings with the sine-kernel density $p(s)=E''(0;s)$, computed with Bornemann's quadrature, and with Poisson; (b) the pair correlation with $1-S(u)^2$ (Montgomery's conjecture); (c) the number variance in windows of length $L$ with the CUE.

**P11.6 (★★★) Statistics application: Keating–Snaith against Selberg.** Selberg's theorem says that $\log|\zeta(\frac12+it)|$, for $t$ near $T$, is approximately Gaussian with variance $\frac12\log\log T$; random matrix theory suggests the finite-$n$ variance $\frac12\sum_{k=1}^n\psi'(k)$ of Exercise 11.8 with $n=\log\frac T{2\pi}$. Sample $\log|Z(t)|$ on $[10^6,10^6+2000]$ and decide which variance fits better. Then compare the empirical moments $\frac1\Delta\int|Z(t)|^{2m}\dd t$, $m=1,2$, with the conjecture $a_m\,\mathbb E|\det(I-U)|^{2m}$ at $n=\log\frac t{2\pi}$, with the arithmetic factor $a_m$ of the chapter.

## Solutions

### P11.1

**How to find it.** $M_D$ has entries $c_{a-b}$ with $c_0=\frac14$ and $c_m=\frac{e^{2\pi im/4}-1}{2\pi im}$. The Poisson binomial law is the coefficient list of $\prod_j(1-p_j+p_jz)$, a polynomial product.

In [2]:
n, alpha, M = 40, 0.25, 20000
m = np.arange(-(n - 1), n)
c = np.where(m == 0, alpha, (np.exp(2j * np.pi * m * alpha) - 1) / (2j * np.pi * np.where(m == 0, 1, m)))
idx = np.arange(n)
MD = c[(idx[:, None] - idx[None, :]) + n - 1]
p = np.linalg.eigvalsh(MD)
print("eigenvalues of M_D: min %.2e, max %.6f; how many in (0.01, 0.99):" % (p.min(), p.max()), ((p > 0.01) & (p < 0.99)).sum())
law = np.array([1.0])
for pj in p: law = np.convolve(law, [1 - pj, pj])
counts = np.array([np.sum(angles(haar(n)) < 2 * np.pi * alpha) for _ in range(M)])
emp = np.bincount(counts, minlength=n + 1) / M
binom = np.array([math.comb(n, k) * alpha**k * (1 - alpha)**(n - k) for k in range(n + 1)])
print(" k   empirical   coin flips   binomial")
for k in range(6, 15): print(f"{k:>2}   {emp[k]:>9.4f}   {law[k]:>10.4f}   {binom[k]:>8.4f}")
print("variance: empirical %.4f, sum p(1-p) %.4f, binomial %.4f" % (counts.var(), np.sum(p * (1 - p)), n * alpha * (1 - alpha)))

eigenvalues of M_D: min -5.37e-16, max 1.000000; how many in (0.01, 0.99): 4


 k   empirical   coin flips   binomial
 6      0.0000       0.0000     0.0530
 7      0.0001       0.0001     0.0857
 8      0.0140       0.0151     0.1179
 9      0.2185       0.2224     0.1397
10      0.5271       0.5248     0.1444
11      0.2239       0.2220     0.1312
12      0.0162       0.0153     0.1057
13      0.0003       0.0001     0.0759
14      0.0000       0.0000     0.0488
variance: empirical 0.5656, sum p(1-p) 0.5687, binomial 7.5000


Of the $40$ coins, almost all are fixed: their parameters are within $0.01$ of $0$ or $1$, and only a handful are genuinely random, which is why the count has variance about $0.57$ against $7.5$ for independent points. The matrix $M_D$ is a discrete version of Slepian's "prolate" time–frequency concentration operator, whose eigenvalues famously jump from $1$ to $0$ in a window of logarithmic width: that is the number variance of Theorem 11.2 seen from the coins.

### P11.2

**How to find it.** Formula (11.1) costs $O(n)$ plus a tail; for the tail $n\sum_{m\ge n}$ use a long finite sum and the bound $\frac1M$ for the rest.

In [3]:
def numvar(n, alpha, M=200000):
    m1 = np.arange(1, n); m2 = np.arange(n, M)
    return 2 / np.pi**2 * (np.sum(np.sin(np.pi * m1 * alpha)**2 / m1) + n * np.sum(np.sin(np.pi * m2 * alpha)**2 / m2**2))
n = 400
print("   L    Var N(D)   log(L)/pi^2   difference   binomial")
for L in (1, 2, 4, 8, 16, 32, 64, 128, 200):
    v = numvar(n, L / n)
    print(f"{L:>4}   {v:>8.4f}   {math.log(L) / math.pi**2:>11.4f}   {v - math.log(L) / math.pi**2:>10.4f}   {L * (1 - L / n):>8.2f}")
L = 32; counts = [np.sum(angles(haar(n)) < 2 * np.pi * L / n) for _ in range(500)]
print("simulation, L = 32: variance %.4f, mean %.3f" % (np.var(counts), np.mean(counts)))

   L    Var N(D)   log(L)/pi^2   difference   binomial
   1     0.3440        0.0000       0.3440       1.00
   2     0.4155        0.0702       0.3452       1.99
   4     0.4861        0.1405       0.3456       3.96
   8     0.5564        0.2107       0.3457       7.84
  16     0.6265        0.2809       0.3455      15.36
  32     0.6959        0.3512       0.3447      29.44
  64     0.7629        0.4214       0.3415      53.76
 128     0.8198        0.4916       0.3281      87.04
 200     0.8369        0.5368       0.3001     100.00


simulation, L = 32: variance 0.6719, mean 31.992


The difference settles near $0.34$; in fact $\operatorname{Var}N=\frac1{\pi^2}(\log 2\pi L+\gamma+1)+o(1)$ for $1\ll L\ll n$, with $\frac{\log2\pi+\gamma+1}{\pi^2}\approx0.345$ (Dyson and Mehta, 1963). An arc expected to hold $200$ eigenvalues has a standard deviation of $0.91$: you can almost predict the count exactly. For independent points it would be $10$. (At $L=n/2$ the arc is half the circle, and symmetry under $D\leftrightarrow D^c$ shows in a slightly smaller variance.)

### P11.3

**How to find it.** Eigenvalues give all the powers at once: $\operatorname{tr}U^j=\sum_ke^{ij\theta_k}$.

In [4]:
n, M = 12, 20000
J = np.arange(1, 31)
T = np.empty((M, len(J)), dtype=complex)
for r in range(M):
    th = angles(haar(n)); T[r] = np.exp(1j * np.outer(J, th)).sum(1)
est = (abs(T)**2).mean(0)
print(" j   E|tr U^j|^2   min(j,n)")
for j in (1, 2, 5, 8, 11, 12, 13, 20, 30): print(f"{j:>2}   {est[j-1]:>11.3f}   {min(j, n):>8}")
x = abs(T[:, 0])**2
print("P(|tr U|^2 > s) against exp(-s):")
for s in (0.5, 1, 2, 3, 4): print(f"  s = {s}:  {np.mean(x > s):.4f}   {math.exp(-s):.4f}")

 j   E|tr U^j|^2   min(j,n)
 1         0.995          1
 2         1.982          2
 5         5.042          5
 8         7.996          8
11        10.901         11
12        12.020         12
13        11.948         12
20        11.829         12
30        11.838         12
P(|tr U|^2 > s) against exp(-s):
  s = 0.5:  0.6066   0.6065
  s = 1:  0.3675   0.3679
  s = 2:  0.1328   0.1353
  s = 3:  0.0479   0.0498
  s = 4:  0.0170   0.0183


The correlation $\min(j,n)$ grows linearly and then saturates exactly at $j=n$: this "ramp and plateau" is the matrix version of Montgomery's $F(\alpha)=\alpha$ for $\alpha<1$ and of his conjecture $F(\alpha)=1$ beyond. The law of $|\operatorname{tr}U|^2$ is already exponential at $n=12$ to within the sampling noise: Diaconis and Shahshahani's theorem has a superexponential rate of convergence (Johansson, 1997).

### P11.4

**How to find it.** For the decomposition, $|\gamma_k|^2$ has tail $(1-x)^{k-1}$, so $|\gamma_k|^2=1-V^{1/(k-1)}$ with $V$ uniform (inverse transform), and the argument is uniform and independent.

In [5]:
def ks_exact(n, m):
    return math.prod((n + l + i) / (l + i) for l in range(m) for i in range(1, m + 1))

def sample_gammas(n, M):
    k = np.arange(1, n + 1)
    r2 = np.where(k == 1, 1.0, 1 - rng.random((M, n))**(1 / np.maximum(k - 1, 1)))
    return np.sqrt(r2) * np.exp(2j * np.pi * rng.random((M, n)))

n, M = 10, 40000
Z = np.array([abs(np.linalg.det(np.eye(n) - haar(n))) for _ in range(M)])
for m in (1, 2): print(f"Haar matrices, n={n}, m={m}: {np.mean(Z**(2*m)):10.2f}   exact {ks_exact(n, m):10.2f}")
for n in (10, 100):
    Zg = abs(np.prod(1 - sample_gammas(n, 200000), axis=1))
    for m in (1, 2, 3):
        print(f"decomposition, n={n:>3}, m={m}: {np.mean(Zg**(2*m)):14.1f}   exact {ks_exact(n, m):14.1f}")

Haar matrices, n=10, m=1:      10.86   exact      11.00
Haar matrices, n=10, m=2:    1486.18   exact    1716.00


decomposition, n= 10, m=1:           11.0   exact           11.0
decomposition, n= 10, m=2:         1649.9   exact         1716.0
decomposition, n= 10, m=3:      1021219.4   exact      1184183.0


decomposition, n=100, m=1:           98.7   exact          101.0
decomposition, n=100, m=2:      4282029.0   exact      9019401.0
decomposition, n=100, m=3: 1267568867725.9   exact 150930018962996.0


For $m=1$ the estimates are within a few percent. For $n=100$, the estimate of the fourth moment is about half the true value, and that of the sixth moment about a hundred times too small (rerun with another seed and these factors jump around wildly, which is itself the point). The reason is the central limit theorem of the chapter: $\log|Z_n|$ is nearly Gaussian with variance $\sigma^2\approx\frac12\log n+0.6$, and $\mathbb E|Z_n|^{2m}=\mathbb Ee^{2m\log|Z_n|}\approx e^{2m^2\sigma^2}$ is carried by values of $\log|Z_n|$ near $2m\sigma^2$, that is $2m\sigma$ standard deviations out: for $n=100$, $\sigma\approx1.7$, and this is about ten standard deviations for $m=3$, a region that $200000$ samples never visit. High moments measure rare events. This is also why the moments of $\zeta$ are hard: they are governed by the rare large values of $\zeta$ on the critical line.

### P11.5

**How to find it.** $Z(t)=2\sum_{k\le N}k^{-1/2}\cos(\vartheta(t)-t\log k)+R$ with $N=\lfloor\sqrt{t/2\pi}\rfloor$ and the first Riemann–Siegel correction $R=(-1)^{N-1}(t/2\pi)^{-1/4}\frac{\cos2\pi(p^2-p-1/16)}{\cos2\pi p}$, $p$ the fractional part of $\sqrt{t/2\pi}$; this is accurate to about $10^{-5}$ at $t=10^6$. A grid of step $0.02$, a fortieth of the mean spacing $0.81$, catches every sign change except for pathologically close pairs, and the count check tells us if one is missed.

In [6]:
def theta(t): return t / 2 * np.log(t / (2 * np.pi)) - t / 2 - np.pi / 8 + 1 / (48 * t) + 7 / (5760 * t**3)

def Zfun(t):
    t = np.atleast_1d(t).astype(float); a = np.sqrt(t / (2 * np.pi)); N = np.floor(a).astype(int); p = a - N
    out = np.empty_like(t)
    for s in range(0, len(t), 4000):
        tt, NN = t[s:s+4000], N[s:s+4000]; k = np.arange(1, NN.max() + 1)
        terms = np.cos(theta(tt)[:, None] - tt[:, None] * np.log(k)[None, :]) / np.sqrt(k)[None, :]
        terms[k[None, :] > NN[:, None]] = 0
        out[s:s+4000] = 2 * terms.sum(1)
    C0 = np.cos(2 * np.pi * (p * p - p - 1 / 16)) / np.cos(2 * np.pi * p)
    return out + (-1.0)**(N - 1) * (t / (2 * np.pi))**(-0.25) * C0

def zeros(T0, T1, h=0.02):
    g = np.arange(T0, T1, h); z = Zfun(g); i = np.nonzero(np.sign(z[:-1]) != np.sign(z[1:]))[0]
    a, b, za = g[i].copy(), g[i + 1].copy(), z[i].copy()
    for _ in range(40):
        mid = (a + b) / 2; zm = Zfun(mid); same = np.sign(zm) == np.sign(za)
        a = np.where(same, mid, a); za = np.where(same, zm, za); b = np.where(same, b, mid)
    return (a + b) / 2

import mpmath
print("check against mpmath.siegelz at t = 10^6 + 0.5:", float(Zfun(1e6 + 0.5)[0]), float(mpmath.siegelz(1e6 + 0.5)))
T0, T1 = 1e6, 1e6 + 5000
g = zeros(T0, T1)
print("zeros found:", len(g), "  Riemann-von Mangoldt main term:", (theta(T1) - theta(T0)) / np.pi)
w = g / (2 * np.pi) * np.log(g / (2 * np.pi * np.e)); sp = np.diff(w)
print("mean rescaled spacing %.5f" % sp.mean())

xg, wg = np.polynomial.legendre.leggauss(40)
def E0(s):
    x = (xg + 1) * s / 2; ww = np.sqrt(wg * s / 2)
    return np.linalg.det(np.eye(40) - ww[:, None] * sinc(x[:, None] - x[None, :]) * ww[None, :])
h = 1e-3
p = lambda s: (E0(s + h) - 2 * E0(s) + E0(s - h)) / h**2
edges = np.linspace(0, 3, 16)
hist, _ = np.histogram(sp, bins=edges); hist = hist / (len(sp) * (edges[1] - edges[0]))
print("(a)   s     zeta    sine kernel   Poisson")
for k in range(15):
    s = (edges[k] + edges[k + 1]) / 2
    print(f"   {s:>5.2f}   {hist[k]:>6.3f}   {p(s):>11.3f}   {math.exp(-s):>7.3f}")
dE = lambda x: (E0(x + h) - E0(x - h)) / (2 * h)
print("small gaps: P(s < 0.25) zeta %.4f, sine kernel %.4f, Poisson %.4f" % (np.mean(sp < 0.25), 1 + dE(0.25), 1 - math.exp(-0.25)))

check against mpmath.siegelz at t = 10^6 + 0.5: -0.9355794715083522 -0.9355806515679347


zeros found: 9533   Riemann-von Mangoldt main term: 9533.48401294524
mean rescaled spacing 0.99998
(a)   s     zeta    sine kernel   Poisson
    0.10    0.035         0.032     0.905
    0.30    0.238         0.263     0.741
    0.50    0.558         0.593     0.607
    0.70    0.862         0.852     0.497
    0.90    0.990         0.933     0.407
    1.10    0.846         0.835     0.333
    1.30    0.647         0.633     0.273
    1.50    0.415         0.415     0.223
    1.70    0.222         0.237     0.183
    1.90    0.108         0.120     0.150
    2.10    0.056         0.054     0.122
    2.30    0.016         0.021     0.100
    2.50    0.006         0.008     0.082
    2.70    0.002         0.002     0.067
    2.90    0.001         0.001     0.055
small gaps: P(s < 0.25) zeta 0.0143, sine kernel 0.0163, Poisson 0.2212


The last line uses $P(s<x)=1+E'(0;x)$ (since $p=E''$ and $E'(0;0)=-1$). All $9533$ zeros predicted by the main term of the Riemann–von Mangoldt formula are found (the remainder $S(T)$ is small here), and the spacing histogram follows the sine-kernel curve to within the sampling noise, with the same deficit of small gaps. Now the pair correlation and the number variance.

In [7]:
du = 0.1; ub = np.arange(0, 3 + du, du)
diffs = []
for k in range(1, 8): diffs.append(w[k:] - w[:-k])
diffs = np.concatenate(diffs)
pc, _ = np.histogram(diffs, bins=ub); pc = pc / (len(w) * du)
print("(b)   u     zeta   1 - S(u)^2")
for k in range(0, 30, 3):
    u = (ub[k] + ub[k + 1]) / 2
    print(f"   {u:>4.2f}   {pc[k]:>6.3f}   {1 - sinc(u)**2:>10.3f}")
print("(c)   L    zeta    CUE (log(2 pi L) + gamma + 1)/pi^2")
for L in (0.5, 1, 2, 4, 8, 16):
    starts = rng.uniform(w[0], w[-1] - L, 20000)
    cnt = np.searchsorted(w, starts + L) - np.searchsorted(w, starts)
    print(f"   {L:>4}   {cnt.var():>5.3f}   {(math.log(2 * math.pi * L) + np.euler_gamma + 1) / math.pi**2:>6.3f}")

(b)   u     zeta   1 - S(u)^2
   0.05    0.007        0.008
   0.35    0.306        0.343
   0.65    0.818        0.810
   0.95    1.035        0.997
   1.25    1.020        0.968
   1.55    0.965        0.959
   1.85    1.012        0.994
   2.15    1.069        0.995
   2.45    0.947        0.984
   2.75    1.023        0.993
(c)   L    zeta    CUE (log(2 pi L) + gamma + 1)/pi^2
    0.5   0.275    0.276
      1   0.326    0.346
      2   0.378    0.416
      4   0.432    0.486
      8   0.434    0.557
     16   0.374    0.627


The pair correlation of the zeros vanishes at $0$ and rises to $1$ exactly as Dyson's function. The number variance agrees with the CUE for windows of one or two zeros, and then the zeros become **more** rigid than the matrices: the variance levels off near $0.42$ instead of continuing its logarithmic growth. Michael Berry explained this saturation in 1988 (*Nonlinearity* 1, 399–407): through the explicit formula, the primes impose order on the zeros at long range, and the saturation level grows only like $\frac1{\pi^2}\log\log T$. This is the first place where the zeta function and the random matrix stop being twins and the arithmetic shows through.

### P11.6

**How to find it.** Sample $\log|Z(t)|$ on a grid finer than the mean spacing. The arithmetic factor is a convergent Euler product: $a_1=1$, and for $m=2$ the local factor is $(1-\frac1p)^4\sum_r(r+1)^2p^{-r}=(1-\frac1p)^4\frac{1+1/p}{(1-1/p)^3}=1-\frac1{p^2}$, so $a_2=\frac6{\pi^2}$.

In [8]:
T0, T1 = 1e6, 1e6 + 2000
tt = np.arange(T0, T1, 0.01)
z = Zfun(tt)
logz = np.log(abs(z))
n_eff = math.log(T0 / (2 * np.pi))
var_rmt = 0.5 * sum(special.polygamma(1, k) for k in range(1, int(round(n_eff)) + 1))
var_rmt_frac = 0.5 * (sum(special.polygamma(1, k) for k in range(1, 12)) + (n_eff - 11) * special.polygamma(1, 12))
print("log T/2pi = %.3f" % n_eff)
print("empirical mean %.4f, empirical variance %.4f" % (logz.mean(), logz.var()))
print("Selberg  1/2 log log T          = %.4f" % (0.5 * math.log(math.log(T0))))
print("RMT      1/2 sum psi'(k), n = %d  = %.4f   (interpolated at n = %.2f: %.4f)" % (round(n_eff), var_rmt, n_eff, var_rmt_frac))
for m in (1, 2):
    a = 1 if m == 1 else 6 / math.pi**2
    emp = np.mean(abs(z)**(2 * m))
    ks = a * np.mean([ks_exact(math.log(t / (2 * math.pi)), m) for t in tt[::1000]])
    print(f"m={m}: empirical moment {emp:9.3f}   Keating-Snaith a_m E|Z_n|^2m {ks:9.3f}")

log T/2pi = 11.978
empirical mean 0.0012, empirical variance 1.9430
Selberg  1/2 log log T          = 1.3129
RMT      1/2 sum psi'(k), n = 12  = 2.0313   (interpolated at n = 11.98: 2.0304)
m=1: empirical moment    13.121   Keating-Snaith a_m E|Z_n|^2m    12.979
m=2: empirical moment  2626.698   Keating-Snaith a_m E|Z_n|^2m  1924.421


Selberg's variance is correct in the limit but off by a large constant at any height we can reach: $\frac12\log\log10^6\approx1.31$, while the zeros give a variance near $1.9$. The random matrix variance with $n=\log\frac T{2\pi}\approx12$ is $2.0$, much closer; Keating and Snaith made exactly this observation in 2000 with Odlyzko's zeros near the $10^{20}$-th, where the discrepancy with Selberg is even more striking. The small remaining gap comes from the primes: the full conjecture of Conrey, Farmer, Keating, Rubinstein and Snaith (2005) multiplies the matrix prediction by arithmetic factors at every order, not only in the leading constant. For the moments, $m=1$ gives $13.12$, matching Hardy–Littlewood's $\log\frac t{2\pi}+2\gamma\approx13.13$, against the matrix value $n+1\approx12.98$; for $m=2$ ($2627$ against $1924$) the leading-order conjecture with $a_2$ is in the right range but lower-order terms, which the matrices do not see, still matter at this height. As a statistician would say: the model is right, and the sample is finite.